# LSC Post-Hoc Contributor Diagnostics

This notebook builds descriptive post-hoc diagnostics for the dissertation's sentiment, intensity, and breadth results. It uses completed LSC outputs and does not rerun Common Crawl processing, VAD matching, frame classification, or embedding models.

The outputs are intended as interpretive diagnostics rather than new inferential tests.

## Setup

The analysis uses three broad publication-year periods: 2014-2017, 2018-2021, and 2022-2026. Target outputs are restricted to ADHD and Autism, with Overall, Clinical, and Lived experience strata.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "paper" / "main.md").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate project root from the current working directory.")


PROJECT_ROOT = find_project_root()

VAD_MATCH_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_collocate_matches.parquet"
BREADTH_CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/breadth/lsc_breadth_sampled_contexts.parquet"
BREADTH_EMBEDDING_PATH = PROJECT_ROOT / "data/interim/lsc/breadth/lsc_breadth_embeddings_normalised.npy"
BREADTH_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/breadth/lsc_breadth_annual_scores.csv"
SENTIMENT_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/sentiment/lsc_sentiment_annual_valence.csv"
INTENSITY_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/intensity/lsc_intensity_annual_arousal.csv"

POSTHOC_DIR = PROJECT_ROOT / "data/processed/lsc/posthoc"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports/tables/lsc/posthoc"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/posthoc"
for directory in [POSTHOC_DIR, REPORT_TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TARGET_UNITS = ["ADHD", "Autism"]
TARGET_ORDER = {unit: i for i, unit in enumerate(TARGET_UNITS)}
REPORT_FRAMES = ["substantive_core_overall", "clinical_only", "lived_only"]
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical",
    "lived_only": "Lived experience",
}
FRAME_ORDER = {frame: i for i, frame in enumerate(REPORT_FRAMES)}

PERIODS = [
    {"period": "2014-2017", "period_start": 2014, "period_end": 2017, "period_order": 1},
    {"period": "2018-2021", "period_start": 2018, "period_end": 2021, "period_order": 2},
    {"period": "2022-2026", "period_start": 2022, "period_end": 2026, "period_order": 3},
]
PERIOD_TABLE = pd.DataFrame(PERIODS)
PERIOD_LABELS = PERIOD_TABLE["period"].tolist()
PERIOD_LOOKUP = {
    year: period["period"]
    for period in PERIODS
    for year in range(period["period_start"], period["period_end"] + 1)
}
PERIOD_ORDER = PERIOD_TABLE.set_index("period")["period_order"].to_dict()
PERIOD_START = PERIOD_TABLE.set_index("period")["period_start"].to_dict()
PERIOD_END = PERIOD_TABLE.set_index("period")["period_end"].to_dict()

CELL_COLUMNS = [
    "analysis_unit",
    "frame_stratum",
    "frame_label",
    "period",
    "period_order",
    "period_start",
    "period_end",
]
CELL_SORT_COLUMNS = ["analysis_order", "frame_order", "period_order"]
CONTENT_POS = {"ADJ", "ADV", "NOUN", "PROPN", "VERB"}
COLLOCATE_TOP_N = 5
BREADTH_TOP_CONTEXTS_PER_CELL = 10
BREADTH_WORD_SOURCE_CONTEXTS_PER_CELL = 20
BREADTH_TOP_WORDS_PER_CELL = 10
FIGURE_TOP_TERMS = 3

CONDITION_COLORS = {"ADHD": "#2F6F9F", "Autism": "#B66A4A"}
GRID_COLOR = "#E7EBEE"

TARGET_WORDS = {
    "ADHD": {"add", "adhd", "attention", "deficit", "hyperactivity", "hyperactive"},
    "Autism": {"autism", "autistic", "asd", "spectrum"},
}
BOILERPLATE_WORDS = {
    "http", "https", "www", "com", "org", "html", "copyright", "reserved", "rights",
    "cookie", "privacy", "policy", "website", "site", "page", "read", "more",
}


def add_period_columns(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["period"] = output["lsc_year"].map(PERIOD_LOOKUP)
    if output["period"].isna().any():
        bad_years = sorted(output.loc[output["period"].isna(), "lsc_year"].dropna().unique().tolist())
        raise ValueError(f"Rows fall outside the configured period bins: {bad_years}")
    output["period_order"] = output["period"].map(PERIOD_ORDER).astype(int)
    output["period_start"] = output["period"].map(PERIOD_START).astype(int)
    output["period_end"] = output["period"].map(PERIOD_END).astype(int)
    return output


def add_display_order(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["frame_label"] = output["frame_stratum"].map(FRAME_LABELS)
    output["analysis_order"] = output["analysis_unit"].map(TARGET_ORDER).astype(int)
    output["frame_order"] = output["frame_stratum"].map(FRAME_ORDER).astype(int)
    return output


def write_table(frame: pd.DataFrame, name: str) -> tuple[Path, Path]:
    data_path = POSTHOC_DIR / name
    report_path = REPORT_TABLE_DIR / name
    frame.to_csv(data_path, index=False)
    frame.to_csv(report_path, index=False)
    return data_path, report_path


def save_lsc_figure(fig: plt.Figure, path: Path) -> tuple[Path, Path]:
    fig.savefig(path, dpi=300, bbox_inches="tight")
    pdf_path = path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight")
    return path, pdf_path


for required_path in [
    VAD_MATCH_PATH,
    BREADTH_CONTEXT_PATH,
    BREADTH_EMBEDDING_PATH,
    BREADTH_ANNUAL_PATH,
    SENTIMENT_ANNUAL_PATH,
    INTENSITY_ANNUAL_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
print(f"Project root: {PROJECT_ROOT}")
print(f"Post-hoc outputs: {POSTHOC_DIR.relative_to(PROJECT_ROOT)}")

Project root: /Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak
Post-hoc outputs: data/processed/lsc/posthoc


## Content-Word VAD Contributors

The main VAD measures intentionally keep stopwords to stay close to the SIBling-style collocate index. These post-hoc tables instead filter to content words so the lists are interpretable. They therefore explain filtered content-word contribution patterns, not the exact all-token VAD aggregate.

In [2]:
def collocate_content_metadata(collocates: pd.Series) -> pd.DataFrame:
    phrases = collocates.dropna().astype(str).drop_duplicates().sort_values().reset_index(drop=True)
    records: list[dict[str, object]] = []
    docs = nlp.pipe((phrase.replace("_", " ") for phrase in phrases), batch_size=1000)
    for phrase, doc in zip(phrases, docs, strict=True):
        content_lemmas: list[str] = []
        for token in doc:
            lemma = token.lemma_.lower().strip()
            if token.is_space or token.is_punct or token.like_num or len(lemma) <= 1:
                continue
            if token.is_stop or lemma in nlp.Defaults.stop_words:
                continue
            if token.pos_ in CONTENT_POS:
                content_lemmas.append(lemma)
        unique_lemmas = list(dict.fromkeys(content_lemmas))
        records.append(
            {
                "collocate": phrase,
                "is_content_collocate": bool(unique_lemmas),
                "content_lemmas": " | ".join(unique_lemmas),
            }
        )
    return pd.DataFrame(records)


vad_matches = pd.read_parquet(VAD_MATCH_PATH)
vad_target = vad_matches.loc[
    vad_matches["analysis_unit"].isin(TARGET_UNITS)
    & vad_matches["frame_stratum"].isin(REPORT_FRAMES)
].copy()
vad_target = add_display_order(add_period_columns(vad_target))

collocate_meta = collocate_content_metadata(vad_target["collocate"])
vad_target = vad_target.merge(collocate_meta, on="collocate", how="left")
content_vad = vad_target.loc[vad_target["is_content_collocate"]].copy()

expected_cells = {
    (unit, frame, period["period"])
    for unit in TARGET_UNITS
    for frame in REPORT_FRAMES
    for period in PERIODS
}
observed_vad_cells = set(
    content_vad[["analysis_unit", "frame_stratum", "period"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_vad_cells = sorted(expected_cells - observed_vad_cells)
if missing_vad_cells:
    raise ValueError(f"Missing content-word VAD cells: {missing_vad_cells}")

cell_totals = (
    content_vad.groupby(CELL_COLUMNS, as_index=False)
    .agg(
        total_filtered_matches_for_cell=("collocate", "size"),
        total_filtered_documents_for_cell=("doc_id", "nunique"),
        filtered_valence_mean=("valence", "mean"),
        filtered_arousal_mean=("arousal", "mean"),
    )
)

collocate_contributions = (
    content_vad.groupby([*CELL_COLUMNS, "collocate", "collocate_type", "content_lemmas"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        documents=("doc_id", "nunique"),
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        weighted_valence_contribution=("valence", "sum"),
        weighted_arousal_contribution=("arousal", "sum"),
    )
    .merge(cell_totals, on=CELL_COLUMNS, how="left")
)
collocate_contributions["match_share"] = (
    collocate_contributions["occurrences"] / collocate_contributions["total_filtered_matches_for_cell"]
)
collocate_contributions["valence_contribution_to_filtered_mean"] = (
    collocate_contributions["weighted_valence_contribution"]
    / collocate_contributions["total_filtered_matches_for_cell"]
)
collocate_contributions["arousal_contribution_to_filtered_mean"] = (
    collocate_contributions["weighted_arousal_contribution"]
    / collocate_contributions["total_filtered_matches_for_cell"]
)
collocate_contributions = add_display_order(collocate_contributions)


def collocate_token_set(value: object) -> set[str]:
    return set(re.findall(r"[a-z]+", str(value or "").lower()))


def target_vocabulary_flags(row: pd.Series) -> pd.Series:
    tokens = collocate_token_set(row["collocate"])
    same_target_words = TARGET_WORDS.get(row["analysis_unit"], set())
    other_target_words = set().union(
        *(words for unit, words in TARGET_WORDS.items() if unit != row["analysis_unit"])
    )
    return pd.Series(
        {
            "same_target_vocabulary_collocate": bool(tokens & same_target_words),
            "other_target_vocabulary_collocate": bool(tokens & other_target_words),
        }
    )


target_flags = collocate_contributions.apply(target_vocabulary_flags, axis=1)
collocate_contributions = pd.concat([collocate_contributions, target_flags], axis=1)


def top_signed_contributors(
    frame: pd.DataFrame,
    value_column: str,
    contribution_column: str,
    positive_label: str,
    negative_label: str,
    n: int = COLLOCATE_TOP_N,
) -> pd.DataFrame:
    sort_columns = ["analysis_order", "frame_order", "period_order", contribution_column]
    positive = (
        frame.loc[frame[contribution_column] > 0]
        .sort_values(sort_columns, ascending=[True, True, True, False])
        .groupby(CELL_COLUMNS, as_index=False)
        .head(n)
        .copy()
    )
    positive["contribution_direction"] = positive_label
    positive["rank"] = positive.groupby(CELL_COLUMNS).cumcount() + 1

    negative = (
        frame.loc[frame[contribution_column] < 0]
        .assign(abs_contribution=lambda data: data[contribution_column].abs())
        .sort_values(["analysis_order", "frame_order", "period_order", "abs_contribution"], ascending=[True, True, True, False])
        .groupby(CELL_COLUMNS, as_index=False)
        .head(n)
        .drop(columns="abs_contribution")
        .copy()
    )
    negative["contribution_direction"] = negative_label
    negative["rank"] = negative.groupby(CELL_COLUMNS).cumcount() + 1

    output = pd.concat([positive, negative], ignore_index=True)
    output = output.sort_values(
        ["analysis_order", "frame_order", "period_order", "contribution_direction", "rank"]
    ).reset_index(drop=True)
    ordered_columns = [
        *CELL_COLUMNS,
        "contribution_direction",
        "rank",
        "collocate",
        "collocate_type",
        "content_lemmas",
        "same_target_vocabulary_collocate",
        "other_target_vocabulary_collocate",
        "occurrences",
        "documents",
        value_column,
        contribution_column,
        f"{value_column}_contribution_to_filtered_mean",
        "match_share",
        "total_filtered_matches_for_cell",
        "total_filtered_documents_for_cell",
        f"filtered_{value_column}_mean",
    ]
    return output[ordered_columns]


sentiment_top = top_signed_contributors(
    collocate_contributions,
    value_column="valence",
    contribution_column="weighted_valence_contribution",
    positive_label="positive",
    negative_label="negative",
)
arousal_top = top_signed_contributors(
    collocate_contributions,
    value_column="arousal",
    contribution_column="weighted_arousal_contribution",
    positive_label="arousal_raising",
    negative_label="arousal_lowering",
)

sentiment_paths = write_table(sentiment_top, "lsc_posthoc_sentiment_content_collocates.csv")
arousal_paths = write_table(arousal_top, "lsc_posthoc_arousal_content_collocates.csv")
cell_total_paths = write_table(
    cell_totals.sort_values(["analysis_unit", "frame_stratum", "period_order"]),
    "lsc_posthoc_vad_content_cell_totals.csv",
)

content_filter_summary = pd.DataFrame(
    [
        {
            "raw_target_vad_rows": len(vad_target),
            "content_word_vad_rows": len(content_vad),
            "content_word_row_share": len(content_vad) / len(vad_target),
            "unique_collocates": vad_target["collocate"].nunique(),
            "unique_content_collocates": int(collocate_meta["is_content_collocate"].sum()),
        }
    ]
)
display(content_filter_summary)
display(sentiment_top.head(12))
display(arousal_top.head(12))
print("Saved VAD contributor tables:")
for output_path in [*sentiment_paths, *arousal_paths, *cell_total_paths]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")

,raw_target_vad_rows,content_word_vad_rows,content_word_row_share,unique_collocates,unique_content_collocates
0,652386,374695,0.574346,10383,9724


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,contribution_direction,rank,collocate,collocate_type,content_lemmas,same_target_vocabulary_collocate,other_target_vocabulary_collocate,occurrences,documents,valence,weighted_valence_contribution,valence_contribution_to_filtered_mean,match_share,total_filtered_matches_for_cell,total_filtered_documents_for_cell,filtered_valence_mean
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,1,disorder,unigram,disorder,False,False,383,336,-0.675000,-258.5250,-0.012184,0.018050,21219,4199,0.062817
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,2,autism,unigram,autism,False,True,485,443,-0.530000,-257.0500,-0.012114,0.022857,21219,4199,0.062817
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,3,depression,unigram,depression,False,False,237,223,-0.938000,-222.3060,-0.010477,0.011169,21219,4199,0.062817
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,4,diagnose,unigram,diagnose,False,False,355,322,-0.455500,-161.7025,-0.007621,0.016730,21219,4199,0.062817
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,5,anxiety,unigram,anxiety,False,False,215,204,-0.708000,-152.2200,-0.007174,0.010132,21219,4199,0.062817
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,1,child,unigram,child,False,False,809,661,0.769000,622.1210,0.029319,0.038126,21219,4199,0.062817
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,2,adult,unigram,adult,False,False,268,203,0.630000,168.8400,0.007957,0.012630,21219,4199,0.062817
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,3,add,unigram,add,True,False,534,453,0.290000,154.8600,0.007298,0.025166,21219,4199,0.062817
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,4,include,unigram,include,False,False,213,207,0.459333,97.8380,0.004611,0.010038,21219,4199,0.062817
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,5,treatment,unigram,treatment,False,False,212,184,0.326000,69.1120,0.003257,0.009991,21219,4199,0.062817


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,contribution_direction,rank,collocate,collocate_type,content_lemmas,same_target_vocabulary_collocate,other_target_vocabulary_collocate,occurrences,documents,arousal,weighted_arousal_contribution,arousal_contribution_to_filtered_mean,match_share,total_filtered_matches_for_cell,total_filtered_documents_for_cell,filtered_arousal_mean
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,1,add,unigram,add,True,False,534,453,-0.105000,-56.070000,-0.002642,0.025166,21219,4199,0.013939
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,2,treatment,unigram,treatment,False,False,212,184,-0.228000,-48.336000,-0.002278,0.009991,21219,4199,0.013939
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,3,patient,unigram,patient,False,False,74,68,-0.614000,-45.436000,-0.002141,0.003487,21219,4199,0.013939
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,4,common,unigram,common,False,False,55,55,-0.746000,-41.030000,-0.001934,0.002592,21219,4199,0.013939
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,5,use,unigram,use,False,False,118,112,-0.336000,-39.648000,-0.001869,0.005561,21219,4199,0.013939
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,1,disorder,unigram,disorder,False,False,383,336,0.530000,202.990000,0.009566,0.018050,21219,4199,0.013939
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,2,anxiety,unigram,anxiety,False,False,215,204,0.730000,156.950000,0.007397,0.010132,21219,4199,0.013939
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,3,hyperactivity,unigram,hyperactivity,True,False,94,90,1.000000,94.000000,0.004430,0.004430,21219,4199,0.013939
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,4,drug,unigram,drug,False,False,121,108,0.542000,65.582000,0.003091,0.005702,21219,4199,0.013939
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,5,issue,unigram,issue,False,False,86,84,0.590000,50.740000,0.002391,0.004053,21219,4199,0.013939


Saved VAD contributor tables:
- data/processed/lsc/posthoc/lsc_posthoc_sentiment_content_collocates.csv
- reports/tables/lsc/posthoc/lsc_posthoc_sentiment_content_collocates.csv
- data/processed/lsc/posthoc/lsc_posthoc_arousal_content_collocates.csv
- reports/tables/lsc/posthoc/lsc_posthoc_arousal_content_collocates.csv
- data/processed/lsc/posthoc/lsc_posthoc_vad_content_cell_totals.csv
- reports/tables/lsc/posthoc/lsc_posthoc_vad_content_cell_totals.csv


## Collocate Contributor Figures

The figures show the top three terms from each saved top-five direction list to keep appendix candidates legible. The CSV tables retain all top-five contributors.

In [3]:
def joined_terms(frame: pd.DataFrame, unit: str, frame_stratum: str, period: str, direction: str, n: int = FIGURE_TOP_TERMS) -> str:
    subset = frame.loc[
        frame["analysis_unit"].eq(unit)
        & frame["frame_stratum"].eq(frame_stratum)
        & frame["period"].eq(period)
        & frame["contribution_direction"].eq(direction)
    ].sort_values("rank")
    terms = subset["collocate"].head(n).tolist()
    return ", ".join(terms) if terms else "-"


def make_contributor_table_figure(
    frame: pd.DataFrame,
    output_path: Path,
    title: str,
    first_direction: str,
    first_label: str,
    second_direction: str,
    second_label: str,
) -> tuple[Path, Path]:
    row_labels: list[str] = []
    table_rows: list[list[str]] = []
    for unit in TARGET_UNITS:
        for frame_stratum in REPORT_FRAMES:
            row_labels.append(f"{unit}\n{FRAME_LABELS[frame_stratum]}")
            row_values = []
            for period in PERIOD_LABELS:
                first_terms = joined_terms(frame, unit, frame_stratum, period, first_direction)
                second_terms = joined_terms(frame, unit, frame_stratum, period, second_direction)
                row_values.append(f"{first_label}: {first_terms}\n{second_label}: {second_terms}")
            table_rows.append(row_values)

    cell_text = [[label, *values] for label, values in zip(row_labels, table_rows, strict=True)]
    col_labels = ["Series", *PERIOD_LABELS]
    fig, ax = plt.subplots(figsize=(14.2, 5.8))
    ax.axis("off")
    table = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="left", colLoc="left")
    table.auto_set_font_size(False)
    table.set_fontsize(8.0)
    table.scale(1.0, 1.65)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#D8DEE3")
        cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor("#EEF2F4")
            cell.set_text_props(weight="bold", color="#263238")
        elif col == 0:
            label = cell.get_text().get_text()
            unit = label.split("\n", 1)[0]
            cell.set_facecolor("#F7F9FA")
            cell.set_text_props(weight="bold", color=CONDITION_COLORS.get(unit, "#263238"))
        else:
            cell.set_facecolor("white")
        if col == 0:
            cell.set_width(0.16)
        else:
            cell.set_width(0.28)

    fig.suptitle(title, x=0.02, y=0.985, ha="left", fontsize=13, fontweight="bold")
    fig.text(
        0.02,
        0.02,
        "Cells show top three report-facing content-word contributors; CSV outputs retain top five per direction.",
        ha="left",
        va="bottom",
        fontsize=8.5,
        color="#54636A",
    )
    return save_lsc_figure(fig, output_path)


sentiment_figure_paths = make_contributor_table_figure(
    sentiment_top,
    FIGURE_DIR / "lsc_posthoc_sentiment_content_collocates.png",
    "Post-hoc sentiment contributors by period and frame",
    first_direction="positive",
    first_label="pos",
    second_direction="negative",
    second_label="neg",
)
arousal_figure_paths = make_contributor_table_figure(
    arousal_top,
    FIGURE_DIR / "lsc_posthoc_arousal_content_collocates.png",
    "Post-hoc arousal contributors by period and frame",
    first_direction="arousal_raising",
    first_label="raise",
    second_direction="arousal_lowering",
    second_label="lower",
)
plt.close("all")

print("Saved collocate contributor figures:")
for output_path in [*sentiment_figure_paths, *arousal_figure_paths]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")

Saved collocate contributor figures:
- reports/figures/lsc/posthoc/lsc_posthoc_sentiment_content_collocates.png
- reports/figures/lsc/posthoc/lsc_posthoc_sentiment_content_collocates.pdf
- reports/figures/lsc/posthoc/lsc_posthoc_arousal_content_collocates.png
- reports/figures/lsc/posthoc/lsc_posthoc_arousal_content_collocates.pdf


## Breadth Context Contributors

Breadth is not a lexical weighted-average measure, so there is no exact collocate analogue. The closest diagnostic is context-level contribution to dispersion: for each target/frame/period cell, this notebook computes each context's average cosine distance to all other contexts in the same cell. High-distance contexts are interpreted as breadth-expanding examples.

In [4]:
def mean_pairwise_cosine_distance(normalised_vectors: np.ndarray) -> float:
    n = normalised_vectors.shape[0]
    if n < 2:
        return float("nan")
    sum_vector = normalised_vectors.sum(axis=0, dtype=np.float64)
    sum_pairwise_similarity = (float(np.dot(sum_vector, sum_vector)) - n) / 2.0
    mean_similarity = 2.0 * sum_pairwise_similarity / (n * (n - 1))
    return float(1.0 - mean_similarity)


def average_distance_to_cell(normalised_vectors: np.ndarray) -> np.ndarray:
    n = normalised_vectors.shape[0]
    if n < 2:
        return np.full(n, np.nan, dtype=float)
    sum_vector = normalised_vectors.sum(axis=0, dtype=np.float64)
    similarity_to_sum = normalised_vectors @ sum_vector
    return 1.0 - ((similarity_to_sum - 1.0) / (n - 1))


def compact_snippet(text: object, width: int = 260) -> str:
    value = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(value) <= width:
        return value
    marker_start = value.find("<t>")
    marker_end = value.find("</t>")
    if marker_start >= 0 and marker_end > marker_start:
        centre = (marker_start + marker_end) // 2
    else:
        centre = len(value) // 2
    left = max(0, centre - width // 2)
    right = min(len(value), left + width)
    left = max(0, right - width)
    snippet = value[left:right].strip()
    if left > 0:
        snippet = "... " + snippet
    if right < len(value):
        snippet = snippet + " ..."
    return snippet


def content_lemmas_for_text(text: object, analysis_unit: str | None = None) -> list[str]:
    cleaned = re.sub(r"</?t>", " ", str(text or ""))
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return []
    excluded = set(BOILERPLATE_WORDS)
    if analysis_unit in TARGET_WORDS:
        excluded.update(TARGET_WORDS[analysis_unit])
    lemmas: list[str] = []
    for token in nlp(cleaned):
        lemma = token.lemma_.lower().strip()
        if token.is_space or token.is_punct or token.like_num or len(lemma) <= 2:
            continue
        if token.is_stop or lemma in nlp.Defaults.stop_words or lemma in excluded:
            continue
        if token.pos_ not in CONTENT_POS:
            continue
        if not re.search(r"[a-z]", lemma):
            continue
        lemmas.append(lemma)
    return lemmas


def top_unique_words(text: object, analysis_unit: str, n: int = 8) -> str:
    counts = Counter(content_lemmas_for_text(text, analysis_unit))
    return ", ".join(word for word, _ in counts.most_common(n))


breadth_contexts = pd.read_parquet(BREADTH_CONTEXT_PATH)
breadth_contexts = breadth_contexts.loc[
    breadth_contexts["analysis_unit"].isin(TARGET_UNITS)
    & breadth_contexts["frame_stratum"].isin(REPORT_FRAMES)
].copy()
breadth_contexts = add_display_order(add_period_columns(breadth_contexts))
breadth_contexts["sample_row_id"] = breadth_contexts["sample_row_id"].astype(int)
breadth_contexts["embedding_row_id"] = breadth_contexts["embedding_row_id"].astype(int)

observed_breadth_cells = set(
    breadth_contexts[["analysis_unit", "frame_stratum", "period"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_breadth_cells = sorted(expected_cells - observed_breadth_cells)
if missing_breadth_cells:
    raise ValueError(f"Missing breadth cells: {missing_breadth_cells}")

embeddings_normalised = np.load(BREADTH_EMBEDDING_PATH, mmap_mode="r")
if breadth_contexts["embedding_row_id"].max() >= embeddings_normalised.shape[0]:
    raise ValueError("Breadth context embedding_row_id exceeds saved embedding matrix rows.")

context_records: list[pd.DataFrame] = []
word_source_records: list[pd.DataFrame] = []
cell_diagnostics: list[dict[str, object]] = []

for group_values, group in breadth_contexts.groupby(CELL_COLUMNS, sort=True):
    group = group.copy().reset_index(drop=True)
    vectors = np.asarray(embeddings_normalised[group["embedding_row_id"].to_numpy(dtype=int)], dtype=np.float32)
    average_distances = average_distance_to_cell(vectors)
    cell_breadth = mean_pairwise_cosine_distance(vectors)
    group["avg_distance_to_cell_contexts"] = average_distances
    group["distance_percentile_in_cell"] = group["avg_distance_to_cell_contexts"].rank(method="average", pct=True)
    std = group["avg_distance_to_cell_contexts"].std(ddof=0)
    group["distance_z_in_cell"] = 0.0 if pd.isna(std) or std == 0 else (
        group["avg_distance_to_cell_contexts"] - group["avg_distance_to_cell_contexts"].mean()
    ) / std
    group["cell_breadth_mean_pairwise_distance"] = cell_breadth
    group["cell_contexts"] = len(group)
    group["cell_documents"] = group["doc_id"].nunique()
    group["cell_domains"] = group["registered_domain"].nunique()

    top_contexts = (
        group.sort_values("avg_distance_to_cell_contexts", ascending=False)
        .head(BREADTH_TOP_CONTEXTS_PER_CELL)
        .copy()
    )
    top_contexts["rank"] = np.arange(1, len(top_contexts) + 1)
    top_contexts["snippet"] = top_contexts["marked_context"].map(compact_snippet)
    top_contexts["content_word_hints"] = [
        top_unique_words(text, unit)
        for text, unit in zip(top_contexts["marked_context"], top_contexts["analysis_unit"], strict=True)
    ]
    context_records.append(top_contexts)

    word_source = (
        group.sort_values("avg_distance_to_cell_contexts", ascending=False)
        .head(BREADTH_WORD_SOURCE_CONTEXTS_PER_CELL)
        .copy()
    )
    word_source_records.append(word_source)

    cell_diagnostics.append(
        {
            "analysis_unit": group_values[0],
            "frame_stratum": group_values[1],
            "frame_label": group_values[2],
            "period": group_values[3],
            "period_order": group_values[4],
            "period_start": group_values[5],
            "period_end": group_values[6],
            "cell_contexts": len(group),
            "cell_documents": group["doc_id"].nunique(),
            "cell_domains": group["registered_domain"].nunique(),
            "cell_breadth_mean_pairwise_distance": cell_breadth,
            "mean_context_avg_distance": float(np.nanmean(average_distances)),
            "max_context_avg_distance": float(np.nanmax(average_distances)),
        }
    )

breadth_context_contributors = pd.concat(context_records, ignore_index=True)
breadth_word_source = pd.concat(word_source_records, ignore_index=True)
breadth_cell_diagnostics = pd.DataFrame(cell_diagnostics)

word_records: list[dict[str, object]] = []
for row in breadth_word_source.itertuples(index=False):
    words = content_lemmas_for_text(row.marked_context, row.analysis_unit)
    for word in words:
        word_records.append(
            {
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "frame_label": row.frame_label,
                "period": row.period,
                "period_order": row.period_order,
                "period_start": row.period_start,
                "period_end": row.period_end,
                "word": word,
                "sample_row_id": row.sample_row_id,
                "doc_id": row.doc_id,
                "avg_distance_to_cell_contexts": row.avg_distance_to_cell_contexts,
            }
        )

breadth_word_summary = pd.DataFrame(word_records)
if breadth_word_summary.empty:
    raise ValueError("No content words were extracted from high-distance breadth contexts.")
word_cell_counts = (
    breadth_word_source.groupby(CELL_COLUMNS, as_index=False)
    .agg(high_distance_contexts_used=("sample_row_id", "nunique"))
)
breadth_word_summary = (
    breadth_word_summary.groupby([*CELL_COLUMNS, "word"], as_index=False)
    .agg(
        occurrences_in_high_distance_contexts=("word", "size"),
        high_distance_contexts_with_word=("sample_row_id", "nunique"),
        high_distance_documents_with_word=("doc_id", "nunique"),
        mean_context_avg_distance=("avg_distance_to_cell_contexts", "mean"),
    )
    .merge(word_cell_counts, on=CELL_COLUMNS, how="left")
)
breadth_word_summary["context_share"] = (
    breadth_word_summary["high_distance_contexts_with_word"]
    / breadth_word_summary["high_distance_contexts_used"]
)
breadth_word_summary = add_display_order(breadth_word_summary)
breadth_word_summary = (
    breadth_word_summary.sort_values(
        [
            "analysis_order",
            "frame_order",
            "period_order",
            "high_distance_contexts_with_word",
            "occurrences_in_high_distance_contexts",
            "word",
        ],
        ascending=[True, True, True, False, False, True],
    )
    .groupby(CELL_COLUMNS, as_index=False)
    .head(BREADTH_TOP_WORDS_PER_CELL)
    .reset_index(drop=True)
)
breadth_word_summary["rank"] = breadth_word_summary.groupby(CELL_COLUMNS).cumcount() + 1

context_columns = [
    *CELL_COLUMNS,
    "rank",
    "lsc_year",
    "doc_id",
    "registered_domain",
    "raw_form",
    "context_source",
    "context_token_count",
    "avg_distance_to_cell_contexts",
    "distance_percentile_in_cell",
    "distance_z_in_cell",
    "cell_breadth_mean_pairwise_distance",
    "cell_contexts",
    "cell_documents",
    "cell_domains",
    "content_word_hints",
    "snippet",
]
breadth_context_contributors = (
    add_display_order(breadth_context_contributors)
    .sort_values(["analysis_order", "frame_order", "period_order", "rank"])
    .reset_index(drop=True)
)
breadth_context_contributors = breadth_context_contributors[context_columns]

word_columns = [
    *CELL_COLUMNS,
    "rank",
    "word",
    "occurrences_in_high_distance_contexts",
    "high_distance_contexts_with_word",
    "high_distance_documents_with_word",
    "high_distance_contexts_used",
    "context_share",
    "mean_context_avg_distance",
]
breadth_word_summary = breadth_word_summary[word_columns]
breadth_cell_diagnostics = breadth_cell_diagnostics.sort_values(["analysis_unit", "frame_stratum", "period_order"])

breadth_context_paths = write_table(
    breadth_context_contributors,
    "lsc_posthoc_breadth_context_contributors.csv",
)
breadth_word_paths = write_table(
    breadth_word_summary,
    "lsc_posthoc_breadth_content_word_summaries.csv",
)
breadth_diagnostic_paths = write_table(
    breadth_cell_diagnostics,
    "lsc_posthoc_breadth_cell_diagnostics.csv",
)

display(breadth_cell_diagnostics.head(9))
display(breadth_context_contributors.head(12))
display(breadth_word_summary.head(12))
print("Saved breadth contributor tables:")
for output_path in [*breadth_context_paths, *breadth_word_paths, *breadth_diagnostic_paths]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")

,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,cell_contexts,cell_documents,cell_domains,cell_breadth_mean_pairwise_distance,mean_context_avg_distance,max_context_avg_distance
0,ADHD,clinical_only,Clinical,2014-2017,1,2014,2017,4495,3123,2340,0.150293,0.150293,0.591353
1,ADHD,clinical_only,Clinical,2018-2021,2,2018,2021,3886,2765,2450,0.139985,0.139985,0.522984
2,ADHD,clinical_only,Clinical,2022-2026,3,2022,2026,3073,2118,1916,0.135732,0.135732,0.441109
3,ADHD,lived_only,Lived experience,2014-2017,1,2014,2017,1085,898,771,0.078423,0.078423,0.407867
4,ADHD,lived_only,Lived experience,2018-2021,2,2018,2021,1124,911,823,0.082683,0.082683,0.509366
5,ADHD,lived_only,Lived experience,2022-2026,3,2022,2026,1194,921,847,0.075956,0.075956,0.569576
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6220,4254,3177,0.136908,0.136908,0.596653
7,ADHD,substantive_core_overall,Overall,2018-2021,2,2018,2021,5626,3937,3426,0.127375,0.127375,0.525702
8,ADHD,substantive_core_overall,Overall,2022-2026,3,2022,2026,4883,3253,2897,0.119728,0.119728,0.589165


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,rank,lsc_year,doc_id,registered_domain,raw_form,context_source,context_token_count,avg_distance_to_cell_contexts,distance_percentile_in_cell,distance_z_in_cell,cell_breadth_mean_pairwise_distance,cell_contexts,cell_documents,cell_domains,content_word_hints,snippet
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,1,2016,36367539aa8db7be,kuscco.com,adhd,target_sentence,33,0.596653,1.000000,7.285519,0.136908,6220,4254,3177,"high, recovered, dairy, prior, liquid, acu, rite, meridia","Although a higher <t>adhd</t> of the recovered dairy from prior others was liquid to 4 acu-rite; meridia of malaria at the online reduction discoloration, this does properly in..."
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,2,2016,80b8d3c69882904e,criminal-record-check.life,adhd,target_sentence,49,0.467933,0.999839,5.245700,0.136908,6220,4254,3177,"tell, prince, likely, possess, bed, census, deliver, mountain","He tells the prince's <t>adhd</t> that he is the likely one who possesses his bed's census, delivered on his mountain check, of casting a franchise bone and persuades them to t..."
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,3,2015,2ac9fd2ba3f1f4a0,illinoistimes.com,adhd,target_sentence,28,0.432776,0.999598,4.688571,0.136908,6220,4254,3177,"production, form, productions, local, actor, jason, goodreau, mac","It’s the first production of the just-formed <t>ADHD</t> Productions, and local actors Jason Goodreau and Mac Warren are performing this must-see play by Richard Dresser."
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,4,2014,5a02a2c8b3617909,illinoistimes.com,adhd,target_sentence,28,0.432776,0.999598,4.688571,0.136908,6220,4254,3177,"production, form, productions, local, actor, jason, goodreau, mac","It’s the first production of the just-formed <t>ADHD</t> Productions, and local actors Jason Goodreau and Mac Warren are performing this must-see play by Richard Dresser."
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,5,2017,7203541365cd664e,additudemag.com,adhd,target_sentence,73,0.420999,0.999357,4.501950,0.136908,6220,4254,3177,"text, buy, thing, middle, world, completely, google, ads",To me the world itself seems to have gone completely <t>ADHD</t> whenever: - Google Ads suggest I buy things I just bought - The phone rings in the middle of a text conversatio...
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6,2014,85efe696f8e2bc12,thewrap.com,adhd,target_sentence,21,0.413120,0.999196,4.377087,0.136908,6220,4254,3177,"network, order, traditional, half, hour, studio, premiere, primetime","The network has also ordered two traditional half-hour shows from the <t>ADHD</t> studio, which will premiere in primetime in 2015."
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,7,2014,612af73ee3c56e54,wikia.com,adhd,target_sentence,24,0.413117,0.999035,4.377050,0.136908,6220,4254,3177,"plugin, edit, architecture, great, joy, acrobat, star, splash",edit Plugin Architecture One of the great joys of Acrobat is staring at the splash screen waiting for countless plugins to load (see <t>ADHD</t>).
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,8,2017,8257571c03571573,theinscribermag.com,adhd,target_sentence,10,0.412968,0.998875,4.374685,0.136908,6220,4254,3177,"problem, company","However, problem has always been that this company has <t>ADHD</t>."
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,9,2016,76d4d3f5e6a0509f,healthyplace.com,adhd,target_sentence,23,0.409417,0.998714,4.318401,0.136908,6220,4254,3177,"article, different, types, educational, assessment, tests, parent, advocate","next: Different Types of Educational Assessment Tests ~ back to Parent Advocate homepage ~ <t>adhd</t> library articles ~ all add/adhd articles APA Reference Staff, H."
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,10,2014,6b893a4bd106a0a4,iss

,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,rank,word,occurrences_in_high_distance_contexts,high_distance_contexts_with_word,high_distance_documents_with_word,high_distance_contexts_used,context_share,mean_context_avg_distance
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,1,disorder,25,10,9,20,0.50,0.381292
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,2,behavior,4,4,4,20,0.20,0.377688
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,3,childhood,4,4,4,20,0.20,0.380967
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,4,impulsivity,4,4,4,20,0.20,0.380967
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,5,inattention,4,4,4,20,0.20,0.380967
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6,case,3,3,3,20,0.15,0.377204
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,7,characterize,3,3,3,20,0.15,0.377204
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,8,diagnose,3,3,3,20,0.15,0.377204
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,9,problem,3,3,3,20,0.15,0.393560
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,10,spectrum,3,3,2,20,0.15,0.383422


Saved breadth contributor tables:
- data/processed/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
- data/processed/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
- data/processed/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv


## Breadth Contributor Figure

The CSV outputs contain top context snippets and top content words. The compact figure shows the top five high-distance content words per target/frame/period cell.

In [5]:
def joined_breadth_words(unit: str, frame_stratum: str, period: str, n: int = 5) -> str:
    subset = breadth_word_summary.loc[
        breadth_word_summary["analysis_unit"].eq(unit)
        & breadth_word_summary["frame_stratum"].eq(frame_stratum)
        & breadth_word_summary["period"].eq(period)
    ].sort_values("rank")
    words = subset["word"].head(n).tolist()
    if not words:
        return "-"
    if len(words) <= 3:
        return ", ".join(words)
    return ", ".join(words[:3]) + "\n" + ", ".join(words[3:])


row_labels = []
table_rows = []
for unit in TARGET_UNITS:
    for frame_stratum in REPORT_FRAMES:
        row_labels.append(f"{unit}\n{FRAME_LABELS[frame_stratum]}")
        table_rows.append([joined_breadth_words(unit, frame_stratum, period) for period in PERIOD_LABELS])

cell_text = [[label, *values] for label, values in zip(row_labels, table_rows, strict=True)]
fig, ax = plt.subplots(figsize=(15.8, 5.6))
ax.axis("off")
table = ax.table(cellText=cell_text, colLabels=["Series", *PERIOD_LABELS], loc="center", cellLoc="left", colLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(8.0)
table.scale(1.0, 1.82)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#D8DEE3")
    cell.set_linewidth(0.6)
    if row == 0:
        cell.set_facecolor("#EEF2F4")
        cell.set_text_props(weight="bold", color="#263238")
    elif col == 0:
        label = cell.get_text().get_text()
        unit = label.split("\n", 1)[0]
        cell.set_facecolor("#F7F9FA")
        cell.set_text_props(weight="bold", color=CONDITION_COLORS.get(unit, "#263238"))
    else:
        cell.set_facecolor("white")
    if col == 0:
        cell.set_width(0.16)
    else:
        cell.set_width(0.28)

fig.suptitle("Post-hoc breadth contributors by period and frame", x=0.02, y=0.985, ha="left", fontsize=13, fontweight="bold")
fig.text(
    0.02,
    0.02,
    "Cells show frequent content words among the top 20 highest-distance contexts; snippet-level context contributors are saved as CSV.",
    ha="left",
    va="bottom",
    fontsize=8.5,
    color="#54636A",
)
breadth_figure_paths = save_lsc_figure(fig, FIGURE_DIR / "lsc_posthoc_breadth_content_words.png")
plt.close("all")

print("Saved breadth contributor figure:")
for output_path in breadth_figure_paths:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")

Saved breadth contributor figure:
- reports/figures/lsc/posthoc/lsc_posthoc_breadth_content_words.png
- reports/figures/lsc/posthoc/lsc_posthoc_breadth_content_words.pdf


## Validation And Handoff

The checks below confirm that all target/frame/period cells are represented and that the post-hoc outputs are aligned with the completed annual analyses.

In [6]:
sentiment_annual = pd.read_csv(SENTIMENT_ANNUAL_PATH)
intensity_annual = pd.read_csv(INTENSITY_ANNUAL_PATH)
breadth_annual = pd.read_csv(BREADTH_ANNUAL_PATH)

for name, frame in [
    ("sentiment annual", sentiment_annual),
    ("intensity annual", intensity_annual),
    ("breadth annual", breadth_annual),
]:
    observed_years = sorted(
        frame.loc[
            frame["analysis_unit"].isin(TARGET_UNITS)
            & frame["frame_stratum"].isin(REPORT_FRAMES),
            "lsc_year",
        ].dropna().unique().astype(int).tolist()
    )
    if observed_years != list(range(2014, 2027)):
        raise ValueError(f"Unexpected year coverage for {name}: {observed_years}")

output_checks = {
    "sentiment_top_rows": len(sentiment_top),
    "arousal_top_rows": len(arousal_top),
    "breadth_context_rows": len(breadth_context_contributors),
    "breadth_word_rows": len(breadth_word_summary),
    "vad_cells": len(observed_vad_cells),
    "breadth_cells": len(observed_breadth_cells),
    "expected_cells": len(expected_cells),
}
if output_checks["vad_cells"] != output_checks["expected_cells"]:
    raise ValueError(output_checks)
if output_checks["breadth_cells"] != output_checks["expected_cells"]:
    raise ValueError(output_checks)

handoff = pd.DataFrame(
    [
        {"artifact": "sentiment content collocates", "rows": len(sentiment_top), "path": sentiment_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "arousal content collocates", "rows": len(arousal_top), "path": arousal_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "VAD content cell totals", "rows": len(cell_totals), "path": cell_total_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth context contributors", "rows": len(breadth_context_contributors), "path": breadth_context_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth content word summaries", "rows": len(breadth_word_summary), "path": breadth_word_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth cell diagnostics", "rows": len(breadth_cell_diagnostics), "path": breadth_diagnostic_paths[0].relative_to(PROJECT_ROOT)},
    ]
)
display(handoff)
print("Validation checks passed.")

,artifact,rows,path
0,sentiment content collocates,180,data/processed/lsc/posthoc/lsc_posthoc_sentiment_content_collocates.csv
1,arousal content collocates,180,data/processed/lsc/posthoc/lsc_posthoc_arousal_content_collocates.csv
2,VAD content cell totals,18,data/processed/lsc/posthoc/lsc_posthoc_vad_content_cell_totals.csv
3,breadth context contributors,180,data/processed/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
4,breadth content word summaries,180,data/processed/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
5,breadth cell diagnostics,18,data/processed/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv


Validation checks passed.
